In [1]:
"""
Step 07: VIX And VIX3M Data Check (shared data layer of the VIX experiments exp08-exp11; roadmap 2026-10-09)

Objective:

    - Check the IBKR VIX and VIX3M bars (store01_rawzone/ibkr_vix_family/, approved by Nicolas on 2026-10-08) through the
      loaders of so/features/vix_features.py, every one cut at 2026-04-15 (the last validation session); nothing dated
      2026-05-14 or later is ever read (the loaders refuse it).
    - Coverage per year; missing, stale and fallback sessions (compared with the known gaps of so/vix_config.py); daily
      close vs last 1-minute close; first and last bar per era; the bar chosen by the _intraday timing; the distribution
      of the six features per timing variant.

Data quality only: no strategy, no label and no return is computed, and nothing is logged in the trial log.
Output: store04_experiments/vix_data_layer/step07_VIX_data_check/ (coverage and disagreement tables).
"""

'\nStep 07: VIX And VIX3M Data Check (shared data layer of the VIX experiments exp08-exp11; roadmap 2026-10-09)\n\nObjective:\n\n    - Check the IBKR VIX and VIX3M bars (store01_rawzone/ibkr_vix_family/, approved by Nicolas on 2026-10-08) through the\n      loaders of so/features/vix_features.py, every one cut at 2026-04-15 (the last validation session); nothing dated\n      2026-05-14 or later is ever read (the loaders refuse it).\n    - Coverage per year; missing, stale and fallback sessions (compared with the known gaps of so/vix_config.py); daily\n      close vs last 1-minute close; first and last bar per era; the bar chosen by the _intraday timing; the distribution\n      of the six features per timing variant.\n\nData quality only: no strategy, no label and no return is computed, and nothing is logged in the trial log.\nOutput: store04_experiments/vix_data_layer/step07_VIX_data_check/ (coverage and disagreement tables).\n'

In [2]:
# IMPORT GENERAL PACKAGES
import numpy as np
import pandas as pd
# IMPORT THE SHARED CONFIGURATION, THE VIX CONSTANTS AND THE PATHS
from so import config, paths, vix_config
# IMPORT DATA LOADING, EXECUTION AND DAILY FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
# IMPORT THE VIX DATA LAYER
from so.features.vix_features import get_vix_daily_feature_dict, get_vix_feature_col_list
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# DISPLAY ALL COLUMNS AND ROWS OF THE TABLES
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)

In [3]:
# DEFINE THE CUTOFF OF EVERY LOAD (LAST VALIDATION SESSION; BEFORE THE UNTOUCHED WINDOW)
DATA_CUTOFF_DATE_STR = "2026-04-15"
# DEFINE THE OUTPUT FOLDER
OUTPUT_PATH_STR = paths.get_experiment_data_path_str("vix_data_layer", "step07_VIX_data_check")
# ASSERT THE CUTOFF IS BEFORE THE UNTOUCHED WINDOW
assert DATA_CUTOFF_DATE_STR < vix_config.UNTOUCHED_START_DATE_STR
# DISPLAY INFORMATION
print(f"Cutoff of every load:\t{DATA_CUTOFF_DATE_STR}\t(untouched window from {vix_config.UNTOUCHED_START_DATE_STR}, never read)")
print(f"Output folder:\t{OUTPUT_PATH_STR}")

Cutoff of every load:	2026-04-15	(untouched window from 2026-05-14, never read)
Output folder:	C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/vix_data_layer/step07_VIX_data_check/


In [4]:
# CALL FUNCTION TO READ THE SPY MINUTE BARS CUT AT THE CUTOFF
complete_ohlcv_pdf = get_complete_ohlcv_pdf(cutoff_date_str_in=DATA_CUTOFF_DATE_STR)
# BUILD THE DAILY SPY TABLE (SESSIONS, DECISION TIMESTAMPS, TRAILING VOLATILITY)
daily_pdf = get_daily_feature_pdf(get_ohlcv_array_dict(complete_ohlcv_pdf))
# ASSERT THE CUT
assert max(daily_pdf["date"]) <= pd.Timestamp(DATA_CUTOFF_DATE_STR).date()
# DISPLAY INFORMATION
print(f"SPY sessions:\t{len(daily_pdf):,}\t{daily_pdf['date'].iloc[0]} -> {daily_pdf['date'].iloc[-1]}\t(last SPY date loaded)")

SPY sessions:	5,353	2005-01-03 -> 2026-04-15	(last SPY date loaded)


In [5]:
# CALL FUNCTION TO LOAD THE VIX AND VIX3M BARS (SAME CUTOFF) AND BUILD THE TABLE AND THE FEATURES OF BOTH VARIANTS
vix_dict = get_vix_daily_feature_dict(daily_pdf, DATA_CUTOFF_DATE_STR, vix_config.VARIANT_STR_LIST)
vix_table_pdf, feature_pdf = vix_dict["vix_table_pdf"], vix_dict["feature_pdf"]
daily_bar_pdf_dict, minute_bar_pdf_dict = vix_dict["daily_bar_pdf_dict"], vix_dict["minute_bar_pdf_dict"]
# DISPLAY THE LAST DATE LOADED OF EVERY SERIES
for key_str, last_date in vix_dict["last_date_dict"].items():
    print(f"Last date loaded:\t{key_str}\t{last_date}")
# DISPLAY THE ROWS AND DATES LOADED
for series_str in vix_config.VIX_SERIES_STR_LIST:
    print(f"{series_str} daily:\t{len(daily_bar_pdf_dict[series_str]):,} rows\t{daily_bar_pdf_dict[series_str]['date'].min()} -> {daily_bar_pdf_dict[series_str]['date'].max()}")
    print(f"{series_str} 1-minute:\t{len(minute_bar_pdf_dict[series_str]):,} rows\t{minute_bar_pdf_dict[series_str]['date'].nunique():,} dates\t{minute_bar_pdf_dict[series_str]['date'].min()} -> {minute_bar_pdf_dict[series_str]['date'].max()}")
# ASSERT THAT NOTHING AFTER THE CUTOFF WAS LOADED
assert all(last_date <= pd.Timestamp(DATA_CUTOFF_DATE_STR).date() for last_date in vix_dict["last_date_dict"].values())

Last date loaded:	vix_daily	2026-04-15
Last date loaded:	vix3m_daily	2026-04-15
Last date loaded:	vix_1min	2026-04-15
Last date loaded:	vix3m_1min	2026-04-15
vix daily:	5,148 rows	2005-10-03 -> 2026-04-15


vix 1-minute:	2,996,586 rows	5,145 dates	2005-10-03 -> 2026-04-15
vix3m daily:	4,193 rows	2009-08-12 -> 2026-04-15
vix3m 1-minute:	1,675,264 rows	4,190 dates	2009-08-12 -> 2026-04-15


In [6]:
"""
Coverage per year (one row per series and year). prev_ok counts the SPY sessions with a usable previous daily close (of
which prev_ok_lagged: the close is 1 to 3 sessions older than the previous SPY session); prev_stale: older than 3
sessions (NaN); prev_missing: no daily bar before the session; prev_before_start: on or before the first date of the
series. intraday_bar / intraday_fallback: the _intraday value came from a 1-minute bar / fell back to _prev.
"""
# DISPLAY THE COVERAGE
coverage_pdf = vix_dict["coverage_pdf"]
display(coverage_pdf)
# DISPLAY THE TOTALS PER SERIES
display(coverage_pdf.groupby("series")[[c for c in coverage_pdf.columns if c not in ["series", "year"]]].sum())

,series,year,session_count,prev_ok,prev_ok_lagged,prev_stale,prev_missing,prev_before_start,intraday_bar,intraday_fallback
0,vix,2005,252,62,0,0,0,190,63,0
1,vix,2006,251,239,4,12,0,0,236,15
2,vix,2007,250,250,0,0,0,0,250,0
3,vix,2008,253,253,0,0,0,0,253,0
4,vix,2009,252,252,0,0,0,0,252,0
5,vix,2010,252,252,0,0,0,0,252,0
6,vix,2011,252,252,1,0,0,0,251,1
7,vix,2012,250,250,0,0,0,0,250,0
8,vix,2013,252,252,0,0,0,0,252,0
9,vix,2014,252,252,0,0,0,0,252,0


,session_count,prev_ok,prev_ok_lagged,prev_stale,prev_missing,prev_before_start,intraday_bar,intraday_fallback
series,,,,,,,,
vix,5353,5151,5,12,0,190,5144,20
vix3m,5353,4193,1,0,0,1160,4190,4


In [7]:
"""
Missing data, compared with the known gaps (so/vix_config.KNOWN_GAP_DICT, from the inventory of 2026-10-08):
the SPY sessions (on or after the first date of the series) without a daily bar of their own date, and without a
1-minute file of their own date.
"""
# COLLECT THE SPY SESSIONS
spy_date_arr = np.array(daily_pdf["date"].tolist())
# LIST TO HOLD THE GAP ROWS
gap_row_list = []
# ITERATE OVER THE SERIES AND BAR SIZES
for series_str in vix_config.VIX_SERIES_STR_LIST:
    for bar_str, bar_pdf in [("daily", daily_bar_pdf_dict[series_str]), ("1min", minute_bar_pdf_dict[series_str])]:
        # COLLECT THE SESSIONS WITHOUT A BAR OF THEIR DATE
        start_date = pd.Timestamp(vix_config.SERIES_START_DATE_DICT[series_str]).date()
        missing_date_list = sorted(set(spy_date_arr[spy_date_arr >= start_date]) - set(bar_pdf["date"]))
        # COLLECT THE KNOWN GAP DATES
        known_date_set = set()
        for gap_start_str, gap_end_str in vix_config.KNOWN_GAP_DICT[(series_str, bar_str)]:
            known_date_set |= {d for d in spy_date_arr if pd.Timestamp(gap_start_str).date() <= d <= pd.Timestamp(gap_end_str).date()}
        # COLLECT THE BAR DATES THAT ARE NOT SPY SESSIONS
        extra_date_list = sorted(set(bar_pdf["date"]) - set(spy_date_arr))
        # ADD THE ROW
        gap_row_list.append({"series": series_str, "bar": bar_str, "missing_count": len(missing_date_list),
                             "known_gap_count": len(known_date_set), "missing_not_known": [str(d) for d in missing_date_list if d not in known_date_set],
                             "known_not_missing": [str(d) for d in sorted(known_date_set) if d not in set(missing_date_list)],
                             "bar_dates_not_spy_sessions": [str(d) for d in extra_date_list]})
# DISPLAY THE GAPS
gap_pdf = pd.DataFrame(gap_row_list)
display(gap_pdf)

,series,bar,missing_count,known_gap_count,missing_not_known,known_not_missing,bar_dates_not_spy_sessions
0,vix,daily,17,17,[],[],[2007-07-02]
1,vix,1min,20,16,"[2020-11-27, 2020-12-24, 2024-07-03, 2024-12-24]",[],[2007-07-02]
2,vix3m,daily,1,1,[],[],[]
3,vix3m,1min,4,4,[],[],[]


In [8]:
# DISPLAY THE STALE SESSIONS AND THE FALLBACK SESSIONS OF EVERY SERIES
for series_str in vix_config.VIX_SERIES_STR_LIST:
    stale_pdf = vix_table_pdf[vix_table_pdf[f"{series_str}_prev_status"].isin(["stale", "missing"])]
    print(f"{series_str}: stale or missing _prev sessions ({len(stale_pdf)}):")
    print(stale_pdf[["date", f"{series_str}_prev_date", f"{series_str}_prev_lag", f"{series_str}_prev_status"]].to_string(index=False) if len(stale_pdf) else "\tnone")
    lagged_pdf = vix_table_pdf[(vix_table_pdf[f"{series_str}_prev_status"] == "ok") & (vix_table_pdf[f"{series_str}_prev_lag"] > 0)]
    print(f"{series_str}: _prev sessions accepted with a lag of 1-3 sessions: {len(lagged_pdf)}; by lag: {lagged_pdf[f'{series_str}_prev_lag'].value_counts().sort_index().to_dict()}")
    fallback_pdf = vix_table_pdf[vix_table_pdf[f"{series_str}_intraday_source"] == "fallback"]
    print(f"{series_str}: _intraday fallbacks ({len(fallback_pdf)}): {[str(d) for d in fallback_pdf['date']]}\n")

vix: stale or missing _prev sessions (12):
      date vix_prev_date  vix_prev_lag vix_prev_status
2006-05-05    2006-04-28             4           stale
2006-05-08    2006-04-28             5           stale
2006-05-09    2006-04-28             6           stale
2006-05-10    2006-04-28             7           stale
2006-05-11    2006-04-28             8           stale
2006-05-12    2006-04-28             9           stale
2006-05-15    2006-04-28            10           stale
2006-05-16    2006-04-28            11           stale
2006-05-17    2006-04-28            12           stale
2006-05-18    2006-04-28            13           stale
2006-05-19    2006-04-28            14           stale
2006-05-22    2006-04-28            15           stale
vix: _prev sessions accepted with a lag of 1-3 sessions: 5; by lag: {1: 3, 2: 1, 3: 1}
vix: _intraday fallbacks (20): ['2006-05-01', '2006-05-02', '2006-05-03', '2006-05-04', '2006-05-05', '2006-05-08', '2006-05-09', '2006-05-10', '2006-05-11

In [9]:
"""
Daily close vs 1-minute closes. The daily close of the index is set after the 16:15 close; the 1-minute session ends at
15:59, 16:14 or 16:29 depending on the era. Two comparisons per date: the last 1-minute bar of the date, and the bar
labelled 16:14 (the minute that ends at 16:15) when it exists. Agreement = absolute difference <= 0.005 index points.
"""
# LIST TO HOLD THE AGREEMENT ROWS AND THE DISAGREEMENTS
agreement_row_list, disagreement_pdf_list = [], []
# ITERATE OVER THE SERIES
for series_str in vix_config.VIX_SERIES_STR_LIST:
    # COLLECT THE LAST 1-MINUTE BAR AND THE 16:14 BAR OF EVERY DATE
    minute_pdf = minute_bar_pdf_dict[series_str]
    last_pdf = minute_pdf.groupby("date").tail(1).set_index("date")
    bar1614_pdf = minute_pdf[pd.DatetimeIndex(minute_pdf["timestamp"]).strftime("%H:%M") == "16:14"].set_index("date")
    # JOIN WITH THE DAILY CLOSES
    compare_pdf = daily_bar_pdf_dict[series_str].set_index("date")[["close"]].rename(columns={"close": "daily_close"})
    compare_pdf = compare_pdf.join(last_pdf[["close", "timestamp"]].rename(columns={"close": "last_1min_close", "timestamp": "last_1min_ts"}), how="inner")
    compare_pdf = compare_pdf.join(bar1614_pdf[["close"]].rename(columns={"close": "close_1614"}), how="left")
    compare_pdf["diff_last"] = compare_pdf["daily_close"] - compare_pdf["last_1min_close"]
    compare_pdf["diff_1614"] = compare_pdf["daily_close"] - compare_pdf["close_1614"]
    # COUNT THE AGREEMENTS
    agree_last_bool = compare_pdf["diff_last"].abs() <= 0.005
    has_1614_bool = compare_pdf["close_1614"].notna()
    agree_1614_bool = compare_pdf["diff_1614"].abs() <= 0.005
    agreement_row_list.append({"series": series_str, "dates_compared": len(compare_pdf), "agree_last_1min": int(agree_last_bool.sum()),
                               "agree_last_share": round(float(agree_last_bool.mean()), 4), "dates_with_1614_bar": int(has_1614_bool.sum()),
                               "agree_1614": int(agree_1614_bool.sum()), "agree_1614_share": round(float(agree_1614_bool[has_1614_bool].mean()), 4) if has_1614_bool.any() else np.nan,
                               "median_abs_diff_last": round(float(compare_pdf["diff_last"].abs().median()), 4),
                               "p99_abs_diff_last": round(float(compare_pdf["diff_last"].abs().quantile(0.99)), 4)})
    # KEEP THE DISAGREEMENTS (LAST 1-MINUTE CLOSE)
    disagreement_pdf_list.append(compare_pdf[~agree_last_bool].reset_index().assign(series=series_str))
# DISPLAY THE AGREEMENT
agreement_pdf = pd.DataFrame(agreement_row_list)
display(agreement_pdf)
# DISPLAY THE AGREEMENT OF THE LAST 1-MINUTE CLOSE PER YEAR
disagreement_pdf = pd.concat(disagreement_pdf_list, ignore_index=True)
display(disagreement_pdf.assign(year=pd.to_datetime(disagreement_pdf["date"]).dt.year).groupby(["series", "year"]).size().rename("disagreement_count").unstack(0).fillna(0).astype(int))
# DISPLAY THE LARGEST DISAGREEMENTS
display(disagreement_pdf.reindex(disagreement_pdf["diff_last"].abs().sort_values(ascending=False).index).head(15))

,series,dates_compared,agree_last_1min,agree_last_share,dates_with_1614_bar,agree_1614,agree_1614_share,median_abs_diff_last,p99_abs_diff_last
0,vix,5144,5010,0.9740,3597,2919,0.8115,0.0,0.02
1,vix3m,4190,3745,0.8938,3464,3023,0.8727,0.0,0.03


series,vix,vix3m
year,,
2005,1,0
2013,0,2
2014,1,1
2016,2,1
2017,0,2
2018,1,1
2021,48,0
2022,58,71
2023,6,124


,date,daily_close,last_1min_close,last_1min_ts,close_1614,diff_last,diff_1614,series
132,2026-01-16,18.84,15.86,2026-01-16 16:59:00-05:00,15.86,2.98,2.98,vix
136,2014-07-17,14.97,12.63,2014-07-17 16:14:00-04:00,12.63,2.34,2.34,vix3m
127,2025-06-18,22.17,20.14,2025-06-18 16:59:00-04:00,20.15,2.03,2.02,vix
126,2025-05-23,20.57,22.29,2025-05-23 16:59:00-04:00,22.32,-1.72,-1.75,vix
112,2023-02-17,21.23,20.02,2023-02-17 16:59:00-05:00,20.00,1.21,1.23,vix
111,2023-01-13,19.49,18.35,2023-01-13 16:59:00-05:00,18.36,1.14,1.13,vix
108,2022-07-01,27.53,26.70,2022-07-01 16:59:00-04:00,26.70,0.83,0.83,vix
128,2025-08-29,16.12,15.36,2025-08-29 16:59:00-04:00,15.38,0.76,0.74,vix
115,2023-09-01,13.82,13.09,2023-09-01 16:59:00-04:00,13.08,0.73,0.74,vix
114,2023-06-16,14.19,13.54,2023-06-16 16:59:00-04:00,13.53,0.65,0.66,vix


In [10]:
"""
First and last bar per era (per year: the most frequent first and last bar label of the 1-minute sessions, and how many
sessions differ from it), and the bar chosen by the _intraday timing: its distance in minutes from the last allowed label
(decision - 1 minute). A distance above 0 means the minute(s) just before the decision were missing.
"""
# LIST TO HOLD THE ERA ROWS
era_row_list = []
# ITERATE OVER THE SERIES
for series_str in vix_config.VIX_SERIES_STR_LIST:
    # COLLECT THE FIRST AND LAST LABEL OF EVERY SESSION
    minute_pdf = minute_bar_pdf_dict[series_str]
    label_pdf = minute_pdf.groupby("date")["timestamp"].agg(["first", "last", "count"])
    label_pdf["first_str"] = pd.DatetimeIndex(label_pdf["first"]).strftime("%H:%M")
    label_pdf["last_str"] = pd.DatetimeIndex(label_pdf["last"]).strftime("%H:%M")
    label_pdf["year"] = pd.to_datetime(label_pdf.index).year
    # SUMMARIZE PER YEAR
    for year_int, year_pdf in label_pdf.groupby("year"):
        first_mode_str, last_mode_str = year_pdf["first_str"].mode().iloc[0], year_pdf["last_str"].mode().iloc[0]
        era_row_list.append({"series": series_str, "year": year_int, "sessions": len(year_pdf), "first_bar_mode": first_mode_str,
                             "first_bar_other": int((year_pdf["first_str"] != first_mode_str).sum()), "last_bar_mode": last_mode_str,
                             "last_bar_other": int((year_pdf["last_str"] != last_mode_str).sum()), "median_bar_count": int(year_pdf["count"].median())})
# DISPLAY THE ERAS
display(pd.DataFrame(era_row_list))
# DISPLAY THE DISTANCE OF THE CHOSEN INTRADAY BAR FROM THE LAST ALLOWED LABEL
for series_str in vix_config.VIX_SERIES_STR_LIST:
    bar_bool_arr = (vix_table_pdf[f"{series_str}_intraday_source"] == "bar").to_numpy()
    allowed_index = pd.DatetimeIndex(vix_table_pdf.loc[bar_bool_arr, "decision_ts"]) - pd.Timedelta(minutes=vix_config.INTRADAY_MIN_LAG_MINUTES)
    distance_arr = ((allowed_index - pd.DatetimeIndex(vix_table_pdf.loc[bar_bool_arr, f"{series_str}_intraday_ts"])).total_seconds() / 60).to_numpy()
    print(f"{series_str}: sessions with an intraday bar {bar_bool_arr.sum():,}; distance 0 min: {int((distance_arr == 0).sum()):,}; 1-15 min: {int(((distance_arr > 0) & (distance_arr <= 15)).sum())}; "
          f"> 15 min: {int((distance_arr > 15).sum())}; max {distance_arr.max():.0f} min; negative: {int((distance_arr < 0).sum())}")
    far_pdf = vix_table_pdf.loc[bar_bool_arr].loc[distance_arr > 15, ["date", "decision_ts", f"{series_str}_intraday_ts"]]
    print(far_pdf.to_string(index=False) if len(far_pdf) else "\t(no chosen bar more than 15 minutes before the allowed label)")

,series,year,sessions,first_bar_mode,first_bar_other,last_bar_mode,last_bar_other,median_bar_count
0,vix,2005,63,09:31,3,15:59,17,389
1,vix,2006,236,09:31,13,15:59,2,389
2,vix,2007,251,09:31,15,15:59,3,389
3,vix,2008,253,09:31,6,15:59,3,389
4,vix,2009,252,09:31,5,15:59,2,389
5,vix,2010,252,09:31,7,15:59,1,389
6,vix,2011,251,09:31,5,15:59,23,389
7,vix,2012,250,09:31,1,16:14,3,404
8,vix,2013,252,09:31,0,16:14,3,404
9,vix,2014,252,09:31,6,16:14,3,404


vix: sessions with an intraday bar 5,144; distance 0 min: 5,144; 1-15 min: 0; > 15 min: 0; max 0 min; negative: 0
	(no chosen bar more than 15 minutes before the allowed label)
vix3m: sessions with an intraday bar 4,190; distance 0 min: 4,190; 1-15 min: 0; > 15 min: 0; max 0 min; negative: 0
	(no chosen bar more than 15 minutes before the allowed label)


In [11]:
"""
Distribution of the six features per timing variant (all sessions to 2026-04-15 where the feature exists), and how
closely the two variants agree (correlation of the same feature under _prev and _intraday). Descriptive only.
"""
# LIST TO HOLD THE DISTRIBUTION TABLES
distribution_pdf_list = []
# ITERATE OVER THE VARIANTS
for variant_str in vix_config.VARIANT_STR_LIST:
    description_pdf = feature_pdf[get_vix_feature_col_list(variant_str)].describe(percentiles=[0.01, 0.2, 0.5, 0.8, 0.99]).T
    description_pdf["first_date"] = [feature_pdf.loc[feature_pdf[c].notna(), "date"].min() for c in description_pdf.index]
    distribution_pdf_list.append(description_pdf)
# DISPLAY THE DISTRIBUTIONS
distribution_pdf = pd.concat(distribution_pdf_list)
display(distribution_pdf.round(4))
# DISPLAY THE AGREEMENT OF THE VARIANTS
display(pd.DataFrame({base_str: {"correlation": feature_pdf[f"{base_str}_prev"].corr(feature_pdf[f"{base_str}_intraday"]),
                                 "both_present": int((feature_pdf[f"{base_str}_prev"].notna() & feature_pdf[f"{base_str}_intraday"].notna()).sum())}
                      for base_str in vix_config.VIX_FEATURE_BASE_STR_LIST}).T.round(4))
# ASSERT THE FADE IS NEVER POSITIVE AND THE PERCENTILE IS IN [0, 1]
for variant_str in vix_config.VARIANT_STR_LIST:
    assert (feature_pdf[f"vix_fade_{variant_str}"].dropna() <= 0).all() and feature_pdf[f"vix_pct250_{variant_str}"].dropna().between(0, 1).all()

,count,mean,std,min,1%,20%,50%,80%,99%,max,first_date
vix_level_prev,5151.0,19.4188,8.6821,9.1400,9.9000,13.1600,17.0700,24.0000,55.8550,82.6900,2005-10-04
vix_pct250_prev,4754.0,0.4678,0.3207,0.0000,0.0000,0.1200,0.4560,0.8080,1.0000,1.0000,2007-05-22
vix_chg5_prev,5141.0,0.0000,0.1504,-0.6287,-0.3478,-0.1046,-0.0096,0.1003,0.4464,1.1407,2005-10-11
vix_fade_prev,5113.0,-0.1745,0.1186,-0.6202,-0.4843,-0.2732,-0.1629,-0.0665,0.0000,0.0000,2005-10-31
vix_term_prev,4193.0,0.8953,0.0741,0.7086,0.7646,0.8360,0.8857,0.9494,1.1436,1.3437,2009-08-13
vrp_prev,5151.0,0.0082,0.0450,-0.6992,-0.1610,0.0028,0.0103,0.0221,0.0835,0.2271,2005-10-04
vix_level_intraday,5153.0,19.4272,8.6599,9.0300,9.9052,13.1700,17.0700,23.9420,55.8716,82.6900,2005-10-03
vix_pct250_intraday,4755.0,0.4681,0.3206,0.0000,0.0000,0.1200,0.4560,0.8120,1.0000,1.0000,2007-05-21
vix_chg5_intraday,5143.0,-0.0000,0.1491,-0.6214,-0.3332,-0.1031,-0.0097,0.1001,0.4379,1.0648,2005-10-10
vix_fade_intraday,5115.0,-0.1732,0.1164,-0.6268,-0.4705,-0.2725,-0.1635,-0.0665,0.0000,0.0000,2005-10-28


,correlation,both_present
vix_level,0.9758,5151.0
vix_pct250,0.9486,4754.0
vix_chg5,0.7444,5141.0
vix_fade,0.8775,5113.0
vix_term,0.9110,4193.0
vrp,0.9460,5151.0


In [12]:
# SAVE THE TABLES (OVERWRITE: THEY ARE REBUILT FROM THE RAW DATA ON EVERY RUN)
write_csv_file_to_path(coverage_pdf, f"{OUTPUT_PATH_STR}vix_coverage_per_year.csv", "W", alert_in=False)
write_csv_file_to_path(agreement_pdf, f"{OUTPUT_PATH_STR}vix_daily_vs_1min_agreement.csv", "W", alert_in=False)
write_csv_file_to_path(disagreement_pdf, f"{OUTPUT_PATH_STR}vix_daily_vs_1min_disagreements.csv", "W", alert_in=False)
write_csv_file_to_path(distribution_pdf.reset_index().rename(columns={"index": "feature"}), f"{OUTPUT_PATH_STR}vix_feature_distribution.csv", "W", alert_in=False)
# DISPLAY INFORMATION
print(f"Saved 4 tables in {OUTPUT_PATH_STR}; last SPY date {daily_pdf['date'].iloc[-1]}; last VIX dates {vix_dict['last_date_dict']}; no trial logged")

Saved 4 tables in C:/Users/nico/Desktop/stock_overflow_data/store04_experiments/vix_data_layer/step07_VIX_data_check/; last SPY date 2026-04-15; last VIX dates {'vix_daily': datetime.date(2026, 4, 15), 'vix3m_daily': datetime.date(2026, 4, 15), 'vix_1min': datetime.date(2026, 4, 15), 'vix3m_1min': datetime.date(2026, 4, 15)}; no trial logged
